In [1]:
import pandas as pd
ratings = pd.read_csv("ml-latest-small/ratings.csv")
movies = pd.read_csv("ml-latest-small/movies.csv")

ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [2]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
print("Ratings:", len(ratings))
print("Users:", ratings["userId"].nunique())
print("Movies rated:", ratings["movieId"].nunique())

print(ratings["rating"].describe())


Ratings: 100836
Users: 610
Movies rated: 9724
count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64


In [4]:
ratings.isnull().sum()
movies.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [6]:
# Given a user and a movie, predict how much that user would like that movie.

# need to split the data intially

from sklearn.model_selection import train_test_split

train, test = train_test_split(
    ratings,
    test_size = 0.2,
    random_state = 42
)

In [ ]:
# making sure that I am in the correct env
import sys
print(sys.executable)

/opt/homebrew/anaconda3/envs/movie-recommender/bin/python


In [ ]:
# getting the unique users and movies

user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId"].unique()

num_users = len(user_ids)
num_movies = len(movie_ids)

print("Users:", num_users)
print("Movies:", num_movies)

Users: 610
Movies: 9724


In [17]:
# converting IDs into mappings for the model
user_to_index = {user_id: i for i, user_id in enumerate(user_ids)}
movie_to_index = {movie_id: i for i, movie_id in enumerate(movie_ids)}

train["user_index"] = train["userId"].map(user_to_index)
train["movie_index"] = train["movieId"].map(movie_to_index)

test["user_index"] = test["userId"].map(user_to_index)
test["movie_index"] = test["movieId"].map(movie_to_index)


In [18]:
# build the reccomendation model

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

Matplotlib is building the font cache; this may take a moment.


In [19]:
class MovieRecommender(keras.Model):
    def __init__(self, num_users, num_movies, embedding_size=50):
        super().__init__()

        self.user_embedding = layers.Embedding(
            input_dim=num_users,
            output_dim=embedding_size
        )

        self.movie_embedding = layers.Embedding(
            input_dim=num_movies,
            output_dim=embedding_size
        )

    def call(self, inputs):
        user_vector = self.user_embedding(inputs[:, 0])
        movie_vector = self.movie_embedding(inputs[:, 1])

        return tf.reduce_sum(user_vector * movie_vector, axis=1)

In [ ]:
# build the model

model = MovieRecommender(
    num_users=num_users,
    num_movies=num_movies
)

In [21]:
# compile model
model.compile(
    optimizer="adam",
    loss="mean_squared_error",
    metrics=["mean_absolute_error"]
)


In [24]:
# prepare training inputs

X_train = train[["user_index", "movie_index"]].values
y_train = train["rating"].values

X_test = test[["user_index", "movie_index"]].values
y_test = test["rating"].values

print(X_train[:5])
print(y_train[:5])

[[ 508 4285]
 [ 325 5629]
 [  56  134]
 [ 609   66]
 [ 461 1172]]
[3. 4. 3. 4. 2.]


In [25]:
# training

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1
)

Epoch 1/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 10.9150 - mean_absolute_error: 3.0747 - val_loss: 4.7997 - val_mean_absolute_error: 1.8486
Epoch 2/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2.6574 - mean_absolute_error: 1.2741 - val_loss: 2.0510 - val_mean_absolute_error: 1.0730
Epoch 3/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 1.3370 - mean_absolute_error: 0.8717 - val_loss: 1.5844 - val_mean_absolute_error: 0.9107
Epoch 4/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.9507 - mean_absolute_error: 0.7391 - val_loss: 1.4266 - val_mean_absolute_error: 0.8543
Epoch 5/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.7645 - mean_absolute_error: 0.6687 - val_loss: 1.3640 - val_mean_absolute_error: 0.8288
Epoch 6/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.6538 - mean_absolute_error: 0.6190 - val_loss: 1.3437 - val_mean_absolute_error: 0.8208
Epoch 7/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.5773 - mean_absolute_er

In [26]:
test_loss, test_mae = model.evaluate(X_test, y_test)

print("Test MAE:", test_mae)

631/631 ━━━━━━━━━━━━━━━━━━━━ 0s 311us/step - loss: 1.3207 - mean_absolute_error: 0.8168
Test MAE: 0.8168204426765442


In [ ]:
# model is trained. now want to start generating movie recs
# lets start with user 1

user_id = 1
user_ratings = ratings[ratings["userId"] == user_id]

user_ratings = user_ratings.merge(
    movies,
    on="movieId"
)

user_ratings.sort_values("rating", ascending=False).head(10)

,userId,movieId,rating,timestamp,title,genres
231,1,5060,5.0,964984002,M*A*S*H (a.k.a. MASH) (1970),Comedy|Drama|War
185,1,2872,5.0,964981680,Excalibur (1981),Adventure|Fantasy
89,1,1291,5.0,964981909,Indiana Jones and the Last Crusade (1989),Action|Adventure
90,1,1298,5.0,964984086,Pink Floyd: The Wall (1982),Drama|Musical
190,1,2948,5.0,964982191,From Russia with Love (1963),Action|Adventure|Thriller
189,1,2947,5.0,964982176,Goldfinger (1964),Action|Adventure|Thriller
188,1,2944,5.0,964981872,"Dirty Dozen, The (1967)",Action|Drama|War
186,1,2899,5.0,964982703,Gulliver's Travels (1939),Adventure|Animation|Children
184,1,2858,5.0,964980868,American Beauty (1999),Drama|Romance
179,1,2700,5.0,964980985,"South Park: Bigger, Longer and Uncut (1999)",Animation|Comedy|Musical


In [37]:
# finding movies 1 has not seen

watched_movies = set(user_ratings["movieId"])

unwatched_movies = movies[
    ~movies["movieId"].isin(watched_movies)
].copy()

print(unwatched_movies.head())

   movieId                               title                      genres
1        2                      Jumanji (1995)  Adventure|Children|Fantasy
3        4            Waiting to Exhale (1995)        Comedy|Drama|Romance
4        5  Father of the Bride Part II (1995)                      Comedy
6        7                      Sabrina (1995)              Comedy|Romance
7        8                 Tom and Huck (1995)          Adventure|Children


In [ ]:
# convert IDs to model indicies -> keep only the movies that the model knows and shows up in rating data
unwatched_movies = unwatched_movies[
    unwatched_movies["movieId"].isin(movie_to_index)
].copy()

unwatched_movies["movie_index"] = (
    unwatched_movies["movieId"].map(movie_to_index)
)

In [ ]:
# get user's model index
user_index = user_to_index[user_id]

In [ ]:
# asking model to predict rating

import numpy as np

user_indices = np.full(
    len(unwatched_movies),
    user_index
)

movie_indices = unwatched_movies["movie_index"].values

prediction_input = np.column_stack(
    (user_indices, movie_indices)
)


In [ ]:
# predictions added to movies 

predictions = model.predict(prediction_input)
unwatched_movies["predicted_rating"] = predictions

297/297 ━━━━━━━━━━━━━━━━━━━━ 0s 252us/step


In [ ]:
# get top 10 recs
# milestone: model is actaully recommending movies

recommendations = unwatched_movies.sort_values(
    "predicted_rating",
    ascending=False
).head(10)

recommendations[
    ["title", "genres", "predicted_rating"]
]

,title,genres,predicted_rating
3505,Phantom of the Paradise (1974),Comedy|Fantasy|Horror|Musical|Thriller,6.536136
1649,Lifeboat (1944),Drama|War,6.453021
2665,Bossa Nova (2000),Comedy|Drama|Romance,5.954045
1111,Kolya (Kolja) (1996),Comedy|Drama,5.926235
7695,"Separation, A (Jodaeiye Nader az Simin) (2011)",Drama,5.918376
7215,Crazy Heart (2009),Drama|Romance,5.896153
924,"Grand Day Out with Wallace and Gromit, A (1989)",Adventure|Animation|Children|Comedy|Sci-Fi,5.893555
5037,Dune (2000),Drama|Fantasy|Sci-Fi,5.868555
5915,Eddie Izzard: Dress to Kill (1999),Comedy,5.862492
2331,Harvey (1950),Comedy|Fantasy,5.831121
